# Module 8 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

In [1]:
from copy import deepcopy
import numpy as np
import random
from typing import List, Dict, Tuple, Callable

## Decision Trees

For this assignment you will be implementing and evaluating a Decision Tree using the ID3 Algorithm (**no** pruning or normalized information gain). Use the provided pseudocode. The data is located at (copy link):

http://archive.ics.uci.edu/ml/datasets/Mushroom

**Just in case** the UCI repository is down, which happens from time to time, I have included the data and name files on Blackboard.

<div style="background: lemonchiffon; margin:20px; padding: 20px;">
    <strong>Important</strong>
    <p>
        No Pandas. The only acceptable libraries in this class are those contained in the `environment.yml`. No OOP, either. You can used Dicts, NamedTuples, etc. as your abstract data type (ADT) for the the tree and nodes.
    </p>
</div>

One of the things we did not talk about in the lectures was how to deal with missing values. There are two aspects of the problem here. What do we do with missing values in the training data? What do we do with missing values when doing classifcation?

There are a lot of different ways that we can handle this.
A common algorithm is to use something like kNN to impute the missing values.
We can use conditional probability as well.
There are also clever modifications to the Decision Tree algorithm itself that one can make.

We're going to do something simpler, given the size of the data set: remove the observations with missing values ("?").

You must implement the following functions:

`train` takes training_data and returns the Decision Tree as a data structure.

```
def train(training_data):
   # returns the Decision Tree.
```

`classify` takes a tree produced from the function above and applies it to labeled data (like the test set) or unlabeled data (like some new data).

```
def classify(tree, observations, labeled=True):
    # returns a list of classifications
```

`evaluate` takes a data set with labels (like the training set or test set) and the classification result and calculates the classification error rate:

$$error\_rate=\frac{errors}{n}$$

Do not use anything else as evaluation metric or the submission will be deemed incomplete, ie, an "F". (Hint: accuracy rate is not the error rate!).

`cross_validate` takes the data and uses 10 fold cross validation (from Module 3!) to `train`, `classify`, and `evaluate`. **Remember to shuffle your data before you create your folds**. I leave the exact signature of `cross_validate` to you but you should write it so that you can use it with *any* `classify` function of the same form (using higher order functions and partial application).

Following Module 3's assignment, `cross_validate` should print out a table in exactly the same format. What you are looking for here is a consistent evaluation metric cross the folds. Print the error rate to 4 decimal places. **Do not convert to a percentage.**

```
def pretty_print_tree(tree):
    # pretty prints the tree
```

This should be a text representation of a decision tree trained on the entire data set (no train/test).

To summarize...

Apply the Decision Tree algorithm to the Mushroom data set using 10 fold cross validation and the error rate as the evaluation metric. When you are done, apply the Decision Tree algorithm to the entire data set and print out the resulting tree.

**Note** Because this assignment has a natural recursive implementation, you should consider using `deepcopy` at the appropriate places.

-----

In [2]:
def create_folds(xs: List, n: int) -> List[List[List]]:
    np.random.shuffle(xs)
    k, m = divmod(len(xs), n)
    # be careful of generators...
    return list(xs[i * k + min(i, m):(i + 1) * k + min(i + 1, m)] for i in range(n))

In [3]:
def create_train_test(folds: List[List[List]], index: int) -> Tuple[List[List], List[List]]:
    training = []
    test = []
    for i, fold in enumerate(folds):
        if i == index:
            test = fold
        else:
            training = training + fold.tolist()
    return np.array(training), np.array(test)

<a id="homogeneous"></a>
## homogeneous

When evaluating whether a decision tree can create a leaf node, or proceed further in branching, it should be checked if it is homogeneous for the target variable. If the dataset is homogeneous, a leaf node can be created and the class label of the dataset can be declared as the class label of the node. If the dataset is not homogeneous, there is a possibility that further branching can used to separate the dataset. This function checks if all of the datapoints in the inputted dataset are of the same label. 

* **data** np.array: dataset where the first column of the array is the label
  
**returns** boolean: true if the dataset is homogeneous, false if not homogeneous

In [4]:
def homogeneous(data):
    value = data[0][0]
    labels = data.T[0]
    for label in labels: 
        if label != value:
            return False
    return True

In [5]:
#test case 1
test_data = np.array([
    ['yellow', 'a', 'b', 'c'],
    ['red', 'a', 'b', 'c'],
    ['yellow', 'a', 'b', 'c'],
])
assert homogeneous(test_data) == False

#test case 2
test_data = np.array([
    ['yellow', 'a', 'b', 'c'],
    ['yellow', 'a', 'b', 'c'],
    ['yellow', 'a', 'b', 'c'],
])
assert homogeneous(test_data) == True

#test case 3
test_data = np.array([
    ['red', 'a', 'b', 'c'],
])

assert homogeneous(test_data) == True

<a id="majority_label"></a>
## majority_label

When creating a decision tree, if there remains no more attributes on which branching can continue, a leaf node must be created. The class label of the leaf node is determined by the majority class label in the dataset at the node. This function finds the modal class label in the dataset.

* **data** np.array: dataset where the first column of the array is the label
  
**returns** string: majority class label of the dataset

In [6]:
def majority_label(data):
    labels = list(data.T[0])
    return max(set(labels), key = labels.count)

In [7]:
#test case 1
test_data = np.array([
    ['yellow', 'a', 'b', 'c'],
    ['red', 'a', 'b', 'c'],
    ['yellow', 'a', 'b', 'c'],
])
assert majority_label(test_data) == 'yellow'

#test case 2
test_data = np.array([
    ['yellow', 'a', 'b', 'c'],
    ['yellow', 'a', 'b', 'c'],
    ['yellow', 'a', 'b', 'c'],
])
assert majority_label(test_data) == 'yellow'

#test case 3
test_data = np.array([
    ['red', 'a', 'b', 'c'],
])
assert majority_label(test_data) == 'red'

<a id="subset"></a>
## subset

When creating a decision tree, as an internal node is created and branched, the data present at the node must be split in regards to the branching rules. The subsetted data is passed through the branch for further branching or creation of a leaf node. This function splits the dataset according a specific attribute, and attribute value. 

* **data** np.array: dataset where the first column of the array is the label
* **attribute** int: index of the attribute to be evaluated
* **element** string: specific value of the attribute to be evaluated
  
**returns** np.array: subset of dataset that meets the criteria of a specific element in a specific attribute

In [8]:
def subset(data, attribute, element):
    subset = []
    for row in data: 
        if row[attribute] == element: 
            subset.append(list(row))

    return np.array(subset)

In [9]:
#test case 1
test_data = np.array([
    ['yellow', 'a', 'b', 'c'],
    ['red', 'a', 'b', 'c'],
    ['yellow', 'x', 'b', 'c'],
])
assert len(subset(test_data, 1, 'a')) == 2

#test case 2
test_data = np.array([
    ['yellow', 'a', 'b', 'c'],
    ['yellow', 'a', 'b', 'c'],
    ['yellow', 'a', 'b', 'c'],
])
assert len(subset(test_data, 1, 'x')) == 0

#test case 3
test_data = np.array([
    ['red', 'a', 'b', 'c']
])

assert list(subset(test_data, 1, 'a')[0]) == ['red','a', 'b', 'c']

<a id="calculate_entropy"></a>
## calculate_entropy

Entropy is the measure of uncertainty or disorder in a dataset. This is calculated by considering the distribution of the class labels in the data.The following formula is used to calculate entropy:

$$ E(S) = -\sum_{i} p_{i}log_{2}(p_{i}) $$ 


When creating a decision tree, the goal is to reduce the entropy of the dataset with branching. In order to evaluate whether the entropy is decreased with branching, the initial entropy of the dataset should be evaluated. This function calculates the entropy of a dataset, in which the first column is the class label.

* **data** np.array: dataset where the first column of the array is the label
  
**returns** float: entropy of the dataset

In [10]:
def calculate_entropy(data):
    labels = list(data.T[0])
    label_names = list(set(labels))

    entropy = 0
    for label in label_names: 
        probability = labels.count(label) / len(labels)
        distribution = probability * np.log2(probability)
        entropy = entropy - distribution
    return entropy

In [11]:
test_data = np.array([
    ['round', 'yellow', 'a'],
    ['square', 'yellow', 'a'],
    ['round', 'yellow', 'a'],
    ['square', 'yellow', 'a'],
])
assert calculate_entropy(test_data) == 1

test_data = np.array([
    ['round', 'yellow', 'a'],
    ['round', 'yellow', 'a'],
    ['round', 'yellow', 'a'],
    ['round', 'yellow', 'a'],
])
assert calculate_entropy(test_data) == 0

test_data = np.array([
    ['round', 'yellow', 'a'],
    ['round', 'yellow', 'a'],
    ['round', 'yellow', 'a'],
    ['square', 'yellow', 'a'],
])
assert 0.8 < calculate_entropy(test_data) < 0.82 # decimal value difficult to equate

<a id="information_gain"></a>
## information_gain

When creating a decision tree, the goal is to reduce the entropy of the dataset with branching. Information gain evaluates the degree to which the entropy of the dataset has reduced. In other words, it evaluates how much information about the dataset is gained. The information gain is calculated using the following formula

$$ G(S, A) = E(S) - \sum_{v \in V_{A}} \frac{|S_{v}|}{|S|} E(S_{v}) $$ 

This function calculates the information gain of a dataset if a particular attribute is used for the branch criterion. 

* **data** np.array: dataset where the first column of the array is the label
* **attribute** int: index of attribute to be evaluated
* **entropy** float: calculated entropy of the dataset before branching
  
**returns** float: information gain of a particular attribute branching on a particular dataset. 

In [12]:
def information_gain(data, attribute, entropy):
    attribute_column = list(data.T[attribute])
    gain = entropy
    for element in list(set(attribute_column)):
        element_distribution = attribute_column.count(element) / len(attribute_column)
        subset_data = subset(deepcopy(data), attribute, element)
        element_entropy = calculate_entropy(subset_data)
        gain = gain - (element_distribution * element_entropy)

    return gain     

In [13]:
test_data = np.array([
    ['round', 'green', 'a'],
    ['square', 'yellow', 'a'],
    ['round', 'green', 'a'],
    ['square', 'yellow', 'a'],
])
assert information_gain(test_data, 1, calculate_entropy(test_data)) == 1

test_data = np.array([
    ['round', 'green', 'a'],
    ['square', 'yellow', 'a'],
    ['round', 'yellow', 'a'],
    ['square', 'green', 'a'],
])
assert information_gain(test_data, 1, calculate_entropy(test_data)) == 0

test_data = np.array([
    ['round', 'green', 'a'],
    ['round', 'yellow', 'a'],
    ['round', 'yellow', 'a'],
    ['square', 'green', 'a'],
])
assert information_gain(test_data, 1, calculate_entropy(test_data)) == calculate_entropy(test_data) - 0.5

<a id="pick_best_attribute"></a>
## pick_best_attribute

When creating a decision tree, the goal is to reduce the entropy of the dataset with branching. Information gain evaluates the degree to which the entropy of the dataset has reduced. In other words, it evaluates how much information about the dataset is gained if a particular attribute is used for branching. The information gain is calculated using the following formula

$$ G(S, A) = E(S) - \sum_{v \in V_{A}} \frac{|S_{v}|}{|S|} E(S_{v}) $$ 

This function calculates the potential information gain of the dataset for each potential attribute that can be used for branching, and chooses the best attribute for the child branch. The best attribute maximises the information gain. 

* **data** np.array: dataset where the first column of the array is the label
* **attributes** List[int]: list of indices of attributes available for branching
  
**returns** int: index of the best attribute for branching chosen by maximum information gain

In [14]:
def pick_best_attribute(data, attributes):
    entropy = calculate_entropy(data)
    information_gain_attribute = []
    for attribute in attributes:
        info_gain = information_gain(data, attribute, entropy)
        information_gain_attribute.append((attribute, info_gain))
    
    information_gain_attribute.sort(reverse=True, key = lambda x: x[1])
    
    return information_gain_attribute[0][0]

In [15]:
test_data = np.array([
    ['round', 'green', 'a'],
    ['square', 'yellow', 'a'],
    ['round', 'green', 'a'],
    ['square', 'yellow', 'a'],
])
assert pick_best_attribute(test_data, [1, 2]) == 1

test_data = np.array([
    ['round', 'green', 'a'],
    ['round', 'blue', 'a'],
    ['round', 'green', 'a'],
    ['square', 'red', 'a'],
])
assert pick_best_attribute(test_data, [1, 2]) == 1

test_data = np.array([
    ['round', 'green', 'a'],
    ['round', 'blue', 'b'],
    ['round', 'green', 'c'],
    ['square', 'yellow', 'd'],
])
assert pick_best_attribute(test_data, [1, 2]) == 1

<a id="id3"></a>
## id3

The ID3 (iterative dichotomiser 3) algorithm is one of the popular algorithm used to generate decision trees. ID3 selects features for branching the data based on the concept of maximising information gain, or reducing the entropy of the data. By reducing the entropy, the uncertainty about a classification prediction also decreases. The function operates by recursively partitioning the dataset until a homogeneous subset of data points belonging to one class is achieved. Each internal node of the decision tree corresponds to an attribute or feature upon which a decision is made. Each leaf node corresponds to a predicted class label that is assigned to a datapoint if the leaf node is reached. 

* **data** np.array: dataset where the first column of the array is the label
* **attributes** List[int]: list of indices of attributes available for branching
* **default** string: default class label if no data is found in a branch.
  
**returns** dict: the recursively produced tree or tree branch

In [16]:
def id3(data, attributes, default):
    #base cases
    if len(data) == 0: 
        return default
    if homogeneous(data): 
        return data[0][0]
    if not attributes:
        return majority_label(data)

    best_attribute = pick_best_attribute(data, attributes)
    
    branch = {best_attribute: {}}
    
    default_label = majority_label(data)
    
    for element in list(set(list(data.T[best_attribute]))):
        subset_data = subset(deepcopy(data), best_attribute, element)
        remaining_attributes = deepcopy(attributes)
        remaining_attributes.remove(best_attribute)
        child = id3(subset_data, remaining_attributes, default_label)
        branch[best_attribute][element] = child
        
    return branch

In [17]:
test_data = np.array([
    ['round', 'green', 'a'],
    ['square', 'yellow', 'a'],
    ['round', 'green', 'a'],
    ['square', 'yellow', 'a'],
])
assert id3(test_data, [1,2], None) == {1: {'green': 'round', 'yellow': 'square'}}

test_data = np.array([
    ['round', 'blue', 'a'],
    ['round', 'blue', 'b'],
    ['round', 'blue', 'a'],
    ['square', 'blue', 'd'],
])
tree = id3(test_data, [1,2], None)
assert len(tree) == 1
assert len(tree[2]) == 3

test_data = np.array([])
assert id3(test_data, [1,2], None) == None

<a id="train"></a>
## train

In the decision tree algorithm, the training of the algorithm is the construction of a decision tree based on the training data. This function initiates the construction of a decision tree based on the inputted data. The function assumes that the first column of the dataset is the label.

* **data** np.array: dataset where the first column of the array is the label
  
**returns** dict: the recursively produced tree by the ID3 algorithm

In [18]:
def train(data):
    attributes = list(range(1, len(data.T)))
    default = majority_label(data)
    tree = id3(data, attributes, default)
    return tree

In [19]:
test_data = np.array([
    ['round', 'green', 'a'],
    ['square', 'yellow', 'a'],
    ['round', 'green', 'a'],
    ['square', 'yellow', 'a'],
])
assert train(test_data) == {1: {'green': 'round', 'yellow': 'square'}}

test_data = np.array([
    ['round', 'blue', 'a'],
    ['round', 'blue', 'b'],
    ['round', 'blue', 'a'],
    ['square', 'blue', 'd'],
])
tree = train(test_data)
assert len(tree) == 1
assert len(tree[2]) == 3

attributes = list(range(1, len(test_data.T)))
assert attributes == [1,2]

<a id="predict"></a>
## predict

Once a decision tree is produced, classification is done by assessing a data point along the constructed tree's rules. Each internal node of the tree corresponds to a feature or attribute of the datapoint. The datapoint follows along the nodes of the tree until a leaf node is reached and a label is assigned. This function classifies a single instance using a decision tree.

* **tree** dict: decision tree
* **instance** list: the instance to be evaluated and classified
  
**returns** string: class label assigned to instance

In [20]:
def predict(tree, instance):
    if not isinstance(tree, dict):
        return tree
    else: 
        feature = next(iter(tree))
        observed = instance[feature]
        if observed in tree[feature]:
            return predict(tree[feature][observed], instance)
        else: 
            return None

In [21]:
test_data = np.array([
    ['round', 'green', 'a'],
    ['square', 'yellow', 'a'],
    ['round', 'green', 'a'],
    ['square', 'yellow', 'a'],
    ['round', 'green', 'a']
])
tree = train(test_data)
assert predict(tree, ['round', 'green', 'a']) == 'round'
assert predict(tree, ['round', 'red', 'a']) == None

test_data = np.array([
    ['round', 'blue', 'a'],
    ['round', 'blue', 'b'],
    ['round', 'blue', 'a'],
    ['square', 'blue', 'd'],
])
tree = train(test_data)
assert predict(tree, ['round', 'red', 'a']) == 'round'

<a id="classify"></a>
## classify

Once a decision tree is produced, classification is done by assessing a data point along the constructed tree's rules. This function takes a test data set and intiates the classification of each instance.

* **tree** dict: decision tree
* **observations** np.array: test data to be classified
* **labeled** boolean: true if the dataset is labelled
  
**returns** list: list of class label predictions assigned to each instance in the test dataset

In [22]:
def classify(tree, observations, labeled=True):
    predictions = []
    for i in observations:
        prediction = predict(tree, list(i))  #assuming first column is actual label
        predictions.append(prediction)
    return predictions
    

In [23]:
test_data = np.array([
    ['round', 'green', 'a'],
    ['square', 'yellow', 'a'],
    ['round', 'green', 'a'],
    ['square', 'yellow', 'a'],
    ['round', 'green', 'a']
])
tree = train(test_data)
observations = np.array([['round', 'green', 'a'], ['round', 'red', 'a']])
assert classify(tree, observations, True) == ['round', None]
assert len(classify(tree, observations, True)) == 2

test_data = np.array([
    ['round', 'blue', 'a'],
    ['round', 'blue', 'b'],
    ['round', 'blue', 'a'],
    ['square', 'blue', 'd'],
])
tree = train(test_data)
observations = np.array([['round', 'red', 'a']])
assert classify(tree, observations, True) == ['round']
assert len(classify(tree, observations, True)) == 1

<a id="evaluate"></a>
## evaluate

Once a decision tree is produced, classification is done by assessing a data point along the constructed tree's rules. The error rate of classification is a metric used to evaluate the efficiency of the program. The error rate can be calculated with the following formula:

$$error\_rate=\frac{errors}{n}$$

This function calculates the error rate of the classification predictions made on a dataset.

* **actuals** list: actual labels of the dataset
* **predictions** list: predicted labels of the dataset

  
**returns** float: calculated error rate of the predictions

In [24]:
def evaluate(actuals, predictions):
    total = len(actuals)
    errors = sum(1 for actual, predicted in zip(actuals, predictions) if actual != predicted)
    error_rate = errors/total
    return round(error_rate, 4)

In [25]:
actual = ['a', 'b', 'c']
prediction = ['a', 'b', 'c']
assert evaluate(actual, prediction) == 0.0

actual = ['a', 'c', 'c', 'd']
prediction = ['a', 'b', 'b', 'd']

assert evaluate(actual, prediction) == 0.5

actual = ['d', 'c', 'c', 'a']
prediction = ['a', 'b', 'b', 'd']
assert evaluate(actual, prediction) == 1.0

<a id="cross_validate"></a>
## cross_validate

Cross validation is a resampling method used to effectively evaluate machine learning models when data availability is limited. The dataset is shuffled and divided into n number of folds. Iteratively, each fold is taken as the test set, while the remaining folds are taken as the training set. The folds are iterated through and fitted and run through the model, n number of times, resulting in n evaluations of the model. 

This function generates 10 folds, iterates through the list of folds and applies the ID3 decision tree algorithm to train and construct a decision tree and predict the test set with it. The error rate is calculated for each fold and averaged across all folds. 

* **data** np.array: dataset where the first column of the array is the label
* **prediction_algorithm** Callable: function used to predict algorithm
  
**returns** float: average error rate across all folds

In [26]:
def cross_validate(data, prediction_algorithm: Callable):
    folds = create_folds(data, 10)

    fold_error, test_error, train_error = [], [], []
    
    for i in range(len(folds)):
        train_data, test_data = create_train_test(folds, i)

        #generate tree and make predictions on the training set and test set
        train_predictions = classify(train(train_data), train_data, labeled = True)
        test_predictions = classify(train(train_data), test_data, labeled = True)

        train_error_rate = evaluate(train_data.T[0], train_predictions)
        test_error_rate = evaluate(test_data.T[0], test_predictions)

        fold_error.append([i, test_error_rate, train_error_rate])
        test_error.append(test_error_rate)
        train_error.append(train_error_rate)
    
    test_average = sum(test_error)/len(test_error)
    train_average = sum(train_error)/len(train_error)
    test_std = (sum([((x - test_average) ** 2) for x in test_error]) / len(test_error)) ** 0.5
    train_std = (sum([((x - train_average) ** 2) for x in train_error]) / len(train_error)) ** 0.5

    print ("{:<8} {:<10} {:<10}".format('Fold','Test','Train'))
    for list in fold_error:
        print("{:<8} {:<10} {:<10}".format(list[0],list[1],list[2]))
    print ("{:<8} {:<10} {:<10}".format('Average', round(test_average, 4), round(train_average, 4)))
    print ("{:<8} {:<10} {:<10}".format('STD', round(test_std, 4), round(train_std,4))) 

    return round(test_average, 4)

In [27]:
test_data = np.array([
    [1, 2, 3],
    [1, 2, 3],
    [1, 2, 3],
    [1, 2, 3],
    [1, 2, 3],
    [1, 2, 3]
])
folds = create_folds(test_data, 3)
assert len(folds) == 3

test_error = [1, 2, 4, 5, 6]
test_average = sum(test_error)/len(test_error) 
assert test_average == 3.6

test_std = (sum([((x - test_average) ** 2) for x in test_error]) / len(test_error)) ** 0.5
assert 1.85 < test_std < 1.86

<a id="pretty_print_tree"></a>
## pretty_print_tree

This function provides a visual representation of the decision tree created. Indentation represents the next depth level of the tree. 

* **tree** dict: decision tree
  
**returns** float: average error rate across all folds

In [28]:
def pretty_print_tree(tree, labels, depth=0, prefix=""):

    indentation = "   " * depth  # Indentation for current depth
    if isinstance(tree, dict):
        for key, value in tree.items():
            if depth % 2 == 0:
                #print name of feature instead of index of feature
                feature_name = labels[int(key)]
                feature = "attribute: "
                print(f"{indentation}{prefix}{feature}{feature_name}")
            else:
                #print values
                print(f"{indentation}{prefix}{key}")
            pretty_print_tree(value, labels, depth + 1, prefix="--> ")
            
    else:
        # leaf node found
        leaf = "class label: "
        print(f"{indentation}{prefix}{leaf}{tree}")

In [29]:
labels = ['label', 'color', 'letter']
test_data = np.array([
    ['round', 'green', 'a'],
    ['square', 'yellow', 'a'],
    ['round', 'green', 'a'],
    ['square', 'yellow', 'a'],
])
tree = train(test_data)
pretty_print_tree(tree, labels, 0, "")

print("\n")

test_data = np.array([
    ['round', 'blue', 'a'],
    ['round', 'blue', 'b'],
    ['round', 'blue', 'a'],
    ['square', 'blue', 'd'],
])
tree = train(test_data)
pretty_print_tree(tree, labels, 0, "")

attribute: color
   --> green
      --> class label: round
   --> yellow
      --> class label: square


attribute: letter
   --> b
      --> class label: round
   --> d
      --> class label: square
   --> a
      --> class label: round


## Dataset

In [30]:
#data loading and cleaning
data = np.loadtxt('agaricus-lepiota.data', delimiter=",", dtype= 'str')

data_cleaned = data[~(data[:,11] == '?'),:] #missing values only found in 11th feature (12th column)

assert len(data) - len(data_cleaned) == 2480  #number of missing values according to dataset information

In [31]:
#cross-validation
mean_error = cross_validate(data_cleaned, classify)

Fold     Test       Train     
0        0.0        0.0       
1        0.0        0.0       
2        0.0        0.0       
3        0.0        0.0       
4        0.0        0.0       
5        0.0        0.0       
6        0.0        0.0       
7        0.0        0.0       
8        0.0        0.0       
9        0.0        0.0       
Average  0.0        0.0       
STD      0.0        0.0       


In [32]:
#construction of tree on all data
tree = train(data)

#feature labels of dataset
labels = ['edibility', 'cap-shape', 'cap_surface', 'cap_color', 'bruises?', 'odor', 'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color', 'stalk-shape', 'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-above-ring', 'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number', 'ring-type', 'spore-print-color', 'population', 'habitat']

#printing of dataset
pretty_print_tree(tree, labels, depth=0, prefix="")

attribute: odor
   --> a
      --> class label: e
   --> c
      --> class label: p
   --> n
      --> attribute: spore-print-color
         --> w
            --> attribute: habitat
               --> w
                  --> class label: e
               --> d
                  --> attribute: gill-size
                     --> b
                        --> class label: e
                     --> n
                        --> class label: p
               --> g
                  --> class label: e
               --> p
                  --> class label: e
               --> l
                  --> attribute: cap_color
                     --> w
                        --> class label: p
                     --> n
                        --> class label: e
                     --> y
                        --> class label: p
                     --> c
                        --> class label: e
         --> k
            --> class label: e
         --> n
            --> class label: e
    

## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.